In [ ]:
import torch_fidelity

def to_uint8_artbench10(x):
    # x is a tensor in [0,1]; scale it to [0,255] and convert it to uint8
    x = x.clamp(0, 1)
    x = (x * 255).to(torch.uint8)
    return x

def collect_real_images_from_loader(loader, max_images=5000):
    xs = []
    count = 0
    for x, _, _ in loader:
        xs.append(x)
        count += x.size(0)
        if count >= max_images:
            break
    x_all = torch.cat(xs, dim=0)[:max_images]
    return x_all

def sample_from_AE(model, loader, images=5000):
    model.eval()
    zs = []
    cont = 0

    with torch.no_grad():

        for x, _, _ in loader:
            x = x.to(device)

            # latent vector
            z = model.encode(x)
            zs.append(z.cpu())
            cont += x.size(0)
            if cont >= images:
                break
    z_all = torch.cat(zs, dim=0)[:images]

    z = z_all[torch.randperm(z_all.size(0))[:images]].to(device)  # shuffle and subset

    with torch.no_grad():
        xhat = model.decode(z).cpu().clamp(0, 1)
    return xhat

def sample_from_VAE(model, images=5000):
    model.eval()
    zs = []
    cont = 0

    with torch.no_grad():

        # random numbers from normal distribution
        z_random = torch.randn(images, model.latent_dim).to(device)

        x_hat = model.decode(z_random).cpu().clamp(0, 1)
    return x_hat

def sample_from_gan(model, num_samples=5000, latent_dim=128):
    model.eval()
    with torch.no_grad():
        # Generate random noise
        noise = torch.randn(num_samples, latent_dim, 1, 1, device=device)

        # Generator
        fake_images = model(noise)

        # Return in [0,1] from [-1,1]
        fake_images = denorm(fake_images).clamp(0,1)

        return fake_images.cpu()

def sample_from_diffusion(unet, vae, scheduler, num_samples=5000, batch_size=64):
    unet.eval()
    vae.eval()
    all_samples = []
    generated_so_far = 0

    # Spatial latent parameters (e.g., 16 channels * 4 * 4 = 256 total)
    latent_channels = 4 
    h_lat, w_lat = 4, 4

    while generated_so_far < num_samples:
        current_batch_size = min(batch_size, num_samples - generated_so_far)
        
        # 4D SHAPE: [Batch, Channels, Height, Width]
        shape = (current_batch_size, latent_channels, h_lat, w_lat)
        
        print(f"Generating Latent batch: {generated_so_far + current_batch_size}/{num_samples}")
        
        with torch.no_grad():
            # 1. Denoising in 4D latent space
            latents = scheduler.p_sample_loop(unet, shape)
            
            # 2. If your VAE decoder expects a flattened vector,
            # you need to do .view(current_batch_size, -1) here.
            # If the decoder is convolutional, keep it as is.
            imgs = vae.decoder(latents) 
            
            imgs = denorm(imgs).cpu().clamp(0, 1)
            all_samples.append(imgs)
            
        generated_so_far += current_batch_size
        torch.cuda.empty_cache()
        
    return torch.cat(all_samples, dim=0)

def sample_from_pixel_diffusion(unet, scheduler, num_samples=5000, image_size=32, batch_size=64):
    """
    Generates images in pixel space using sub-batches to avoid CUDA Out of Memory errors.
    """
    unet.eval()
    all_samples = []
    generated_so_far = 0

    while generated_so_far < num_samples:
        # Compute how many samples to generate in this step
        current_batch_size = min(batch_size, num_samples - generated_so_far)
        shape = (current_batch_size, 3, image_size, image_size)
        
        print(f"Generating batch: {generated_so_far + current_batch_size}/{num_samples}")
        
        with torch.no_grad():
            # Generate the sub-batch using your GaussianDiffusion class
            batch_images = scheduler.p_sample_loop(unet, shape)
            # Denormalize and move to CPU to free GPU memory immediately
            batch_images = denorm(batch_images).cpu().clamp(0, 1)
            all_samples.append(batch_images)
            
        generated_so_far += current_batch_size
        # Manually clear GPU cache between batches
        torch.cuda.empty_cache()
        
    # Concatenate all sub-batches into one final tensor
    return torch.cat(all_samples, dim=0)

In [ ]:
import tempfile
from pathlib import Path
from PIL import Image

# save images to disk for FID calculation
def save_images_for_fid(images, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for i in range(images.size(0)):
        img = images[i].permute(1, 2, 0).cpu().numpy() 
         # CxHxW -> HxWxC
        Image.fromarray(img).save(out_dir/f'{i:05d}.png')

def compute_fid(real_batch_u8, gen_batch_u8):
    with tempfile.TemporaryDirectory() as tmpdir:
        real_dir = Path(tmpdir) / 'real'
        gen_dir = Path(tmpdir) / 'gen'
        save_images_for_fid(real_batch_u8, real_dir)
        save_images_for_fid(gen_batch_u8, gen_dir)
        metrics = torch_fidelity.calculate_metrics(
            input1=str(real_dir), 
            input2=str(gen_dir), 
            cuda=torch.cuda.is_available(), 
            fid=True, 
            kid=True,
            kid_subset_size=100,
            kid_subset=50, 
            verbose=False)
    return float(metrics['frechet_inception_distance']), float(metrics['kernel_inception_distance_mean'])

In [ ]:
final_train_loader, final_test_loader = build_artbench_loaders(
    train_hf,
    test_hf,
    batch_size=BATCH_SIZE,
    use_gan_norm=False,
    train_ids=None
)

print(f"Training images: {len(final_train_loader.dataset)}")

# I load the best model

# Once finished, I compute FID using the TEST SET as reference

In [ ]:
# Try 10 different seeds
N_REPETITIONS = 10
N_FID_SAMPLES = 5000

# List for storing results for every seed
conv_fid_scores = [] 
vae_fid_scores = []
conv_kid_scores = [] 
vae_kid_scores = []

# Lists for results
results = {
    'AE': {'fid': [], 'kid': []},
    'VAE': {'fid': [], 'kid': []},
    'GAN': {'fid': [], 'kid': []},
    'Diffusion': {'fid': [], 'kid': []}
}

print(f"Starting evaluation protocol (10 repetitions), {N_FID_SAMPLES} samples for FID/KID calculation each time...")

real_images = collect_real_images_from_loader(final_test_loader, max_images=N_FID_SAMPLES)
real_uint8 = to_uint8_artbench10(real_images)

for i in range(N_REPETITIONS):
    # Change seed at every iteration for variability
    torch.manual_seed(SEED + i)

    # Prepare real and generated samples

    real = collect_real_images_from_loader(train_loader_from_csv, max_images=N_FID_SAMPLES)
    conv_gen = sample_from_AE(convolutionalAE, train_loader_from_csv, images=N_FID_SAMPLES)
    vae_gen = sample_from_VAE(convolutionalVAE, images=N_FID_SAMPLES)

    # Conversion to uint8 for FID calculation
    real_uint8 = to_uint8_artbench10(real)
    conv_uint8 = to_uint8_artbench10(conv_gen)
    vae_uint8 = to_uint8_artbench10(vae_gen)

    f_conv, k_conv = compute_fid(real_uint8, conv_uint8)
    f_vae, k_vae = compute_fid(real_uint8, vae_uint8)

    conv_fid_scores.append(f_conv)
    conv_kid_scores.append(k_conv)
    vae_fid_scores.append(f_vae)
    vae_kid_scores.append(k_vae)

    torch.cuda.empty_cache()  # free GPU memory after each iteration

    print(f"Repetition {i+1} completed.")

# --- FINAL RESULTS (Mean ± Std) ---
print("-" * 30)
print(f"FINAL RESULTS (ArtBench-10, n={N_FID_SAMPLES})")
print(f"Conv AE FID: {np.mean(conv_fid_scores):.4f} ± {np.std(conv_fid_scores):.4f}")
print(f"Conv AE KID: {np.mean(conv_kid_scores):.4f} ± {np.std(conv_kid_scores):.4f}")
print("-" * 30)
print(f"VAE FID:     {np.mean(vae_fid_scores):.4f} ± {np.std(vae_fid_scores):.4f}")
print(f"VAE KID:     {np.mean(vae_kid_scores):.4f} ± {np.std(vae_kid_scores):.4f}")